> 在线讲义：[第 12 章 · 网络形状：深宽、专家粒度、残差与多 token 预测](https://wikieden.github.io/ai-infra-from-constraints/lessons/12-network-shape.html) · 本书《从约束推导 AI Infra》

# 第 12 章 · 网络形状：深宽、专家粒度、残差与多 token 预测

配套讲义：仓库里 `ai-infra/notes/lessons/12-network-shape.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 深宽 | $Ld^2$ | `dense_scale(4, 0.5)` | 1.0（参数相近，层数 ×4） |
| 第二步 三种粒度 | $3Edf$、$6kdf$ | `expert_totals(128, 1536, 8)` 等 | 参数、运算三组相同；16 token × 94 层 ≈ 454 GFLOPs |
| 第三步 Sinkhorn | 行归一、列归一 | `sinkhorn([[1, 3], [1, 1]], 1)` | 和表里一致；迭代 50 轮行列和都≈1 |
| 第四步 推测解码 | $(T_s + T_v)/\bar g$ | `spec_per_output(30, g)` | 15、20、30 ms；打平点 1.5 |

Qwen3-235B-A22B（94 层、d = 4096、128 个专家宽 1536、选 8 个）与推测解码的 20 ms / 30 ms 取自 AII 第 2 章 §2.5；
深宽比例、Sinkhorn 的 2×2 矩阵是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `dense_scale`**：稠密主干参数 ∝ L·d²：层数乘 a、宽度乘 b，参数变成 a·b² 倍。

In [ ]:
def dense_scale(layer_factor: float, width_factor: float) -> float:
    """稠密主干参数 ∝ L·d²：层数乘 a、宽度乘 b，参数变成 a·b² 倍。"""
    return layer_factor * width_factor**2

**函数 `expert_totals`**：单层路由专家 (总参数 3Edf, 单 token 运算 6kdf)。

In [ ]:
def expert_totals(E: int, f: int, k: int, d: int = 4096) -> tuple[int, int]:
    """单层路由专家 (总参数 3Edf, 单 token 运算 6kdf)。"""
    return 3 * E * d * f, 6 * k * d * f

**函数 `sinkhorn`**：Sinkhorn 迭代：交替把每行、每列除以各自的和。

In [ ]:
def sinkhorn(m: list[list[float]], rounds: int) -> list[list[float]]:
    """Sinkhorn 迭代：交替把每行、每列除以各自的和。"""
    for _ in range(rounds):
        m = [[v / sum(r) for v in r] for r in m]  # 每行除以行和
        cols = [sum(r[j] for r in m) for j in range(len(m[0]))]
        m = [[r[j] / cols[j] for j in range(len(r))] for r in m]  # 每列除以列和
    return m

**函数 `spec_per_output`**：推测解码每个输出分摊的时间：(T_s + T_v) / 平均有效输出数。

In [ ]:
def spec_per_output(t_round: float, g: float) -> float:
    """推测解码每个输出分摊的时间：(T_s + T_v) / 平均有效输出数。"""
    return t_round / g

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：层数 ×4、宽度 ×1/2（编的）")
print(f"  参数变成 {dense_scale(4, 0.5):.2f} 倍，依赖链变成 4 倍")

In [ ]:
print("\n第二步：Qwen3-235B-A22B 的三种专家粒度")
for E, f, k in ((128, 1536, 8), (64, 3072, 4), (256, 768, 16)):
    p, op = expert_totals(E, f, k)
    print(f"  E = {E:3d}，f = {f}，选 {k:2d}：每层参数 {p:,}，单 token 运算 {op:,}")
print(f"  16 个 token × 94 层 = {16 * 94 * expert_totals(128, 1536, 8)[1] / 1e9:.0f} GFLOPs")

In [ ]:
print("\n第三步：Sinkhorn（编的 2×2）")
m1 = sinkhorn([[1, 3], [1, 1]], 1)
print(f"  一轮后 {[[round(v, 3) for v in r] for r in m1]}，行和 {[round(sum(r), 3) for r in m1]}")
m50 = sinkhorn([[1, 3], [1, 1]], 50)
print(f"  50 轮后行和 {[round(sum(r), 4) for r in m50]}，列和 {[round(sum(r[j] for r in m50), 4) for j in range(2)]}")

In [ ]:
print("\n第四步：推测解码（普通 decode 20 ms，一轮 30 ms）")
for g in (2, 1.5, 1):
    print(f"  平均有效输出 {g} → 每输出 {spec_per_output(30, g):.0f} ms")
print(f"  打平点 g = 30 / 20 = {30 / 20}")

## 3. 代码里才看得到的

1. 一轮 Sinkhorn 后列和精确为 1，但行和又被打乱了（0.933、1.067）；要多轮才同时接近 1。
2. `expert_totals` 只算路由专家，不含路由器和共享专家；原文说路由器参数随专家数变，所以三种粒度的整层参数并不完全相同。
3. `spec_per_output` 里 $\bar g$ 是平均值；一轮至少出 1 个有效 token（验证时主模型自己的预测总能用）。

## 4. 自己改一改

- 把 `sinkhorn` 的轮数改成 1、3、10、50，打印行和与列和，看收敛速度。
- 换一个 4×4 的正矩阵试 Sinkhorn。
- 用 `spec_per_output` 画出 $\bar g$ 从 1 到 4 的分摊时间曲线。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 平房和高楼，面积一样楼层不同；
> 专家切大切小，总量不变形状变；
> 残差留原料，几路就留几份；
> 草稿审得过，才比一字一字写得快。